# Semantic Search System with Fuzzy Clustering and Semantic Cache

This notebook implements a lightweight semantic search system using
the 20 Newsgroups dataset.

Core components:

1. Vector embeddings
2. FAISS vector search
3. Fuzzy clustering using Gaussian Mixture Models
4. Semantic cache built from scratch
5. FastAPI service layer

In [3]:
!pip install sentence-transformers faiss-cpu scikit-learn matplotlib numpy pandas tf-keras

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------ --------------------- 0.8/1.7 MB 8.0 MB/s eta 0:00:01
   ------------------------ --------------- 1.0/1.7 MB 9.9 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 3.0 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.mixture import GaussianMixture
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

import faiss

In [3]:
# Path to the raw 20 Newsgroups dataset.
# The dataset is structured as directories where each folder
# represents a newsgroup category and each file is a single post.

DATASET_PATH = "../data/20_newsgroups"

docs = []
labels = []
categories = []

for category in os.listdir(DATASET_PATH):

    category_path = os.path.join(DATASET_PATH, category)

    if os.path.isdir(category_path):

        categories.append(category)

        for file in os.listdir(category_path):

            file_path = os.path.join(category_path, file)

            try:
                with open(file_path, "r", encoding="latin1") as f:
                    docs.append(f.read())
                    labels.append(category)
            except:
                continue

print("Total documents:", len(docs))
print("Total categories:", len(categories))
print("Categories:", categories[:10])

Total documents: 19997
Total categories: 20
Categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball']


## Observations from Raw Documents

The Usenet articles contain extensive metadata headers including:

- Xref
- Path
- From
- Newsgroups
- Subject
- Message-ID
- Organization
- Date

These fields describe message routing and email metadata rather
than the semantic content of the discussion.

Including these tokens would introduce noise into the embedding
space and may cause the model to learn artifacts rather than
true topic relationships.

Therefore the preprocessing step removes headers and retains
only the message body before generating embeddings.

### Dataset Structure

The 20 Newsgroups corpus is distributed as a directory
structure where each folder represents a topic category
and each file is an individual Usenet message.

Instead of using the sklearn loader, the raw dataset
is processed directly to allow explicit inspection
and cleaning of the message format.

This approach makes it possible to analyze the
metadata headers present in Usenet posts and
remove them before generating embeddings.

### Header Removal Strategy

Inspection of the raw documents shows that each Usenet post
begins with a metadata header containing routing information
such as `From`, `Subject`, `Organization`, and `Message-ID`.

These fields describe message transmission rather than
the semantic content of the discussion.

To ensure embeddings represent the actual message meaning,
the preprocessing step removes the header block and retains
only the message body.

In [4]:
def remove_headers(text):

    # Usenet posts begin with a metadata header block
    # containing routing information such as:
    # From, Subject, Organization, Message-ID, etc.
    #
    # These fields do not represent the semantic topic
    # of the message and can introduce noise into
    # embedding generation.
    #
    # The message body starts after the first blank line.
    
    parts = text.split("\n\n", 1)

    if len(parts) > 1:
        return parts[1]

    return text


# Apply header removal to all documents
clean_docs = [remove_headers(d) for d in docs]

## Embedding Generation

To perform semantic search and clustering, each document must be
converted into a numerical vector representation.

Traditional methods such as TF-IDF represent text using sparse
word-frequency vectors, which capture lexical similarity but
struggle with semantic relationships.

Instead, this system uses a pretrained transformer-based model
to generate dense semantic embeddings.

### Model Choice: all-MiniLM-L6-v2

This model from the SentenceTransformers library was chosen because:

• It produces 384-dimensional dense embeddings  
• It is lightweight and fast enough for ~20k documents  
• It performs well on semantic similarity tasks  
• It can run locally without requiring large GPU resources

In [5]:
# Load a pretrained transformer model for generating semantic embeddings.
# The 'all-MiniLM-L6-v2' model from SentenceTransformers is used because
# it provides a strong balance between performance and computational cost.

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Embedding Generation

To perform semantic search and clustering, each document must be
converted into a numerical vector representation.

Traditional approaches such as TF-IDF represent text using sparse
word-frequency vectors. While effective for keyword matching,
they often fail to capture semantic similarity between documents.

Instead, this system uses dense semantic embeddings generated by
a pretrained transformer model from the SentenceTransformers library.

### Model Choice

The model **all-MiniLM-L6-v2** was selected because:

• It produces 384-dimensional dense embeddings  
• It is lightweight and computationally efficient  
• It performs well on semantic similarity tasks  
• It can embed ~20k documents quickly on CPU

These embeddings allow documents with similar meanings to
appear close together in vector space, enabling semantic search
and clustering.

In [6]:
# Generate semantic embeddings for each cleaned document.
# Each document will be mapped to a 384-dimensional vector
# representing its semantic meaning.

embeddings = model.encode(
    clean_docs,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

C:\ProgramData\anaconda3\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Embedding shape: (19997, 384)


### Embedding Representation

Each document in the corpus is represented as a
384-dimensional dense vector.

Documents discussing similar topics will produce
embeddings that are closer together in vector space.

This representation enables downstream tasks such as:

• semantic search  
• fuzzy clustering  
• similarity-based caching

In [8]:
import numpy as np

# We will save embeddings to disk so they do not need to be recomputed
# when restarting the notebook or building the API layer later.

np.save("../artifacts/document_embeddings.npy", embeddings)